In [1]:
%load_ext autoreload
%autoreload 2

# WW3 boundaries: real ERA5 data, end to end

Same wiring as `ww3_boundaries_demo.ipynb` (synthetic `reference_waves`), swapped to the real product: `ww3_obc_product_name="era5_wave_spectra"`, `ww3_obc_function_name="get_era5_2d_spectra"`. MOM6's own IC/OBC still use the fast `reference_ocean` product -- only the wave boundary is real here.

**Before running:** `era5_wave_spectra` needs a real Copernicus CDS API key (see `get_era5_2d_spectra`'s description in `raw_data_access/datasets/era5.py` for account setup -- register, accept the `reanalysis-era5-complete` license, write `~/.cdsapirc_era5`). Each of the 36 GET requests below (4 boundaries x 9 days) queues separately on CDS's MARS backend -- expect anywhere from single-digit minutes to a few hours *per request* the first time you run this for a new domain/date range, not seconds. `fetch_raw_chunk`'s per-day-per-boundary caching means every chunk only ever needs to be fetched once: once `extract_forcings/ww3/raw_data/` is populated for a given boundary+date, re-running this notebook (or any other case reusing the same bounding box and dates) skips straight to REGRID.

## Step 0: Paths and imports

Follows the "Fast CrocoDash Case Setup" pattern from the repo's `CLAUDE.md` -- same domain as the synthetic-data demo, so a real ERA5 raw fetch cached for one case is reusable by the other.

In [2]:
from pathlib import Path

from CrocoDash.grid import Grid
from CrocoDash.topo import Topo
from CrocoDash.vgrid import VGrid
from CrocoDash.case import Case

CESMROOT = Path("~/work/installs/full_regional_cesm").expanduser()
CASENAME = "ww3_boundaries_demo_real_era5_manual"
CASEROOT = (Path("~/croc_cases") / CASENAME).expanduser()
INPUTDIR = (Path("~/scratch/croc_input") / CASENAME).expanduser()

print(CESMROOT)
print(CASEROOT)
print(INPUTDIR)

/glade/u/home/manishrv/work/installs/full_regional_cesm
/glade/u/home/manishrv/croc_cases/ww3_boundaries_demo_real_era5_manual
/glade/u/home/manishrv/scratch/croc_input/ww3_boundaries_demo_real_era5_manual


## Step 1: Grid, topography, vertical grid

The exact domain from CLAUDE.md's fast-setup example (10x10 degrees at 1-degree resolution, flat 1000m bathymetry, uniform vgrid).

In [3]:
grid = Grid(
    lenx=10.0,
    leny=10.0,
    resolution=1.0,
    xstart=-60.0,
    ystart=30.0,
    name=CASENAME,
)
topo = Topo(grid, min_depth=10.0, git=False)
topo.set_flat(1000.0)
vgrid = VGrid.uniform(nk=10, depth=1000.0, name=CASENAME)

## Step 2: Create the case

Same `WW3_SESP` swap as the synthetic-data demo.

In [4]:
case = Case(
    cesmroot=CESMROOT,
    caseroot=CASEROOT,
    inputdir=INPUTDIR,
    compset="1850_DATM%NYF_SLND_SICE_MOM6%REGIONAL_SROF_SGLC_WW3_SESP",
    ocn_grid=grid,
    ocn_topo=topo,
    ocn_vgrid=vgrid,
    atm_grid_name="T62",
    machine="derecho",
    project="ncgd0011",
    override=True,
)
print("WW3 in compset:", case.ww3_in_compset)

INFO:	initialize:Initializing the visualCaseGen system...
INFO:  csp_solver:CspSolver initialized.


Creating case...

Running the create_newcase tool with the following command:

/glade/u/home/manishrv/work/installs/full_regional_cesm/cime/scripts/create_newcase --compset 1850_DATM%NYF_SLND_SICE_MOM6%REGIONAL_SROF_SGLC_WW3_SESP --res USER_RES --case /glade/u/home/manishrv/croc_cases/ww3_boundaries_demo_real_era5_manual --machine derecho --run-unsupported --project ncgd0011 --non-local 

The create_newcase command was successful.

• Updating case xml variables to include newly generated ocean grid "ww3_boundaries_demo_real_era5_manual" with the following properties:
 nx: 10, ny: 10. ocean mesh: /glade/u/home/manishrv/scratch/croc_input/ww3_boundaries_demo_real_era5_manual/ocean/ESMF_mesh_ww3_boundaries_demo_real_era5_manual_5f52d9.nc.

./xmlchange OCN_NX=10 --non-local

./xmlchange OCN_NY=10 --non-local

./xmlchange OCN_DOMAIN_MESH=/glade/u/home/manishrv/scratch/croc_input/ww3_boundaries_demo_real_era5_manual/ocean/ESMF_mesh_ww3_boundaries_demo_real_era5_manual_5f52d9.nc --non-local



INFO:	stage:SUCCESS: All stages are complete.


Case created successfully at /glade/u/home/manishrv/croc_cases/ww3_boundaries_demo_real_era5_manual.

To further customize, build, and run the case, navigate to the case directory in your terminal. To create another case, restart the notebook.

./xmlchange MOM6_MEMORY_MODE=dynamic_symmetric --non-local

The following additional configuration options are required to run and must be provided with any listed arguments in configure_forcings:
  - WW3: boundaries
  - conditions: boundaries, product_name, function_name, compset
WW3 in compset: True


## (Optional) Seed a previously-fetched raw ERA5 cache

`Case(..., override=True)` above wipes/recreates `inputdir` -- any raw fetch cache from a *previous case* does NOT survive it, even for the same bounding box/dates. `fetch_raw_chunk`'s existence check only helps *within* one case's lifetime (re-running this notebook without re-creating the case), not across a fresh `override=True` case. If you've already paid the CDS queue cost once (e.g. from an earlier run of this same domain/date range) and saved the raw chunks somewhere, copy them into `INPUTDIR / "extract_forcings" / "ww3" / "raw_data"` **after** case creation and **before** `process_forcings()` to skip straight to REGRID. Set `SOURCE_CACHE` to `None` to fetch for real instead -- the path below is this exact domain/date range's own already-fetched cache from the synthetic-data demo's case (`ww3_boundaries_demo`), so it works out of the box; if that case ever gets cleaned up, this cell just no-ops (0 chunks seeded) and `process_forcings()` below fetches live instead.

In [13]:
import shutil

# Point this at a directory of previously-fetched
# "{boundary}_unprocessed.{date}_{date}.nc" files for this exact bounding
# box + date range, or set to None to fetch live from CDS.
SOURCE_CACHE = Path("~/scratch/croc_input/ww3_boundaries_demo/extract_forcings/ww3/raw_data").expanduser()

if SOURCE_CACHE is not None:
    raw_dir = INPUTDIR / "extract_forcings" / "ww3" / "raw_data"
    raw_dir.mkdir(parents=True, exist_ok=True)
    n = 0
    for f in Path(SOURCE_CACHE).glob("*_unprocessed.*.nc"):
        shutil.copy(f, raw_dir / f.name)
        n += 1
    print(f"seeded {n} cached raw ERA5 chunks into {raw_dir}")

seeded 36 cached raw ERA5 chunks into /glade/u/home/manishrv/scratch/croc_input/ww3_boundaries_demo_real_era5_manual/extract_forcings/ww3/raw_data


## Step 3: Configure + process forcings

MOM6's IC/OBC stay on `reference_ocean` (fast, deterministic -- no reason to make the ocean side slow too). WW3's OBC now points at the real `era5_wave_spectra` product: `process_forcings()` fetches 2D wave spectra from Copernicus CDS for every boundary/day, decodes the raw GRIB directly via `eccodes`, and keeps every real ERA5 grid point in each boundary's buffered window as its own station for `ww3_bounc`'s linear interpolation.

In [12]:
case.configure_forcings(
    date_range=["2020-01-01 00:00:00", "2020-01-09 00:00:00"],
    product_name="reference_ocean",
    function_name="get_reference_ocean_data",
    ww3_obc_product_name="era5_wave_spectra",
    ww3_obc_function_name="get_era5_2d_spectra",
)

INFO:CrocoDash.forcing_configurations.base:[SKIP] tides missing args: ['tpxo_elevation_filepath', 'tpxo_velocity_filepath', 'tidal_constituents']
INFO:CrocoDash.forcing_configurations.base:[SKIP] BGC incompatible with compset
INFO:CrocoDash.forcing_configurations.base:[REQUIRED] Activating WW3
INFO:CrocoDash.forcing_configurations.base:[SKIP] CICE incompatible with compset
INFO:CrocoDash.forcing_configurations.base:[SKIP] BGCIC incompatible with compset
INFO:CrocoDash.forcing_configurations.base:[SKIP] BGCIronForcing incompatible with compset
INFO:CrocoDash.forcing_configurations.base:[SKIP] BGCRiverNutrients incompatible with compset
INFO:CrocoDash.forcing_configurations.base:[SKIP] Runoff incompatible with compset
INFO:CrocoDash.forcing_configurations.base:[SKIP] Chl missing args: ['chl_processed_filepath']
INFO:CrocoDash.forcing_configurations.base:[REQUIRED] Activating conditions
INFO:CrocoDash.forcing_configurations.base:Configuring WW3


./xmlchange WW3_GRID_INP_DIR=/glade/u/home/manishrv/scratch/croc_input/ww3_boundaries_demo_real_era5_manual/wave --non-local

./xmlchange HIST_OPTION=nhours --non-local

./xmlchange HIST_N=1 --non-local



INFO:CrocoDash.forcing_configurations.base:Configuring conditions


Adding parameter changes to user_nl_mom:

  ! Initial conditions
  INIT_LAYERS_FROM_Z_FILE = True
  Z_INIT_ALE_REMAPPING = True
  TEMP_SALT_INIT_VERTICAL_REMAP_ONLY = True
  DEPRESS_INITIAL_SURFACE = True
  VELOCITY_CONFIG = file
  TEMP_SALT_Z_INIT_FILE = init_tracers.nc
  SURFACE_HEIGHT_IC_FILE = init_eta.nc
  VELOCITY_FILE = init_vel.nc
  Z_INIT_FILE_PTEMP_VAR = temp
  Z_INIT_FILE_SALT_VAR = salt
  SURFACE_HEIGHT_IC_VAR = eta_t
  U_IC_VAR = u
  V_IC_VAR = v

  ! Open boundary conditions
  OBC_NUMBER_OF_SEGMENTS = 4
  OBC_FREESLIP_VORTICITY = False
  OBC_FREESLIP_STRAIN = False
  OBC_COMPUTED_VORTICITY = True
  OBC_COMPUTED_STRAIN = True
  OBC_ZERO_BIHARMONIC = True
  OBC_TRACER_RESERVOIR_LENGTH_SCALE_OUT = 3.0E+04
  OBC_TRACER_RESERVOIR_LENGTH_SCALE_IN = 3000.0
  BRUSHCUTTER_MODE = True
  OBC_SEGMENT_001 = "J=0,I=0:N,FLATHER,ORLANSKI,NUDGED,ORLANSKI_TAN,NUDGED_TAN"
  OBC_SEGMENT_001_VELOCITY_NUDGING_TIMESCALES = 0.3, 360.0
  OBC_SEGMENT_001_DATA = "U=file:forcing_obc_segment_001.nc(u

In [11]:
case.process_forcings()

INFO:CrocoDash.extract_forcings.obc:GET [south]: 2020-01-01 → 2020-01-09
INFO:CrocoDash.extract_forcings.obc:GET [north]: 2020-01-01 → 2020-01-09
INFO:CrocoDash.extract_forcings.obc:GET [west]: 2020-01-01 → 2020-01-09
INFO:CrocoDash.extract_forcings.obc:GET [east]: 2020-01-01 → 2020-01-09
INFO:CrocoDash.extract_forcings.obc:REGRID [south]: 9-day slices
INFO:CrocoDash.extract_forcings.obc:Saved regridded file as forcing_obc_segment_001_2020-01-01_2020-01-09.nc
INFO:CrocoDash.extract_forcings.obc:REGRID [north]: 9-day slices


No arakawa_grid provided, assuming the variable mapping for your data product is already in correct format.
Arakawa A grid detected in variable mapping
No arakawa_grid provided, assuming the variable mapping for your data product is already in correct format.
Arakawa A grid detected in variable mapping


INFO:CrocoDash.extract_forcings.obc:Saved regridded file as forcing_obc_segment_002_2020-01-01_2020-01-09.nc
INFO:CrocoDash.extract_forcings.obc:REGRID [west]: 9-day slices
INFO:CrocoDash.extract_forcings.obc:Saved regridded file as forcing_obc_segment_003_2020-01-01_2020-01-09.nc
INFO:CrocoDash.extract_forcings.obc:REGRID [east]: 9-day slices


No arakawa_grid provided, assuming the variable mapping for your data product is already in correct format.
Arakawa A grid detected in variable mapping
No arakawa_grid provided, assuming the variable mapping for your data product is already in correct format.
Arakawa A grid detected in variable mapping


INFO:CrocoDash.extract_forcings.obc:Saved regridded file as forcing_obc_segment_004_2020-01-01_2020-01-09.nc
INFO:CrocoDash.extract_forcings.obc:MERGE [south]
INFO:CrocoDash.extract_forcings.obc:Merged file forcing_obc_segment_001.nc already exists. Skipping.
INFO:CrocoDash.extract_forcings.obc:MERGE [north]
INFO:CrocoDash.extract_forcings.obc:Merged file forcing_obc_segment_002.nc already exists. Skipping.
INFO:CrocoDash.extract_forcings.obc:MERGE [west]
INFO:CrocoDash.extract_forcings.obc:Merged file forcing_obc_segment_003.nc already exists. Skipping.
INFO:CrocoDash.extract_forcings.obc:MERGE [east]
INFO:CrocoDash.extract_forcings.obc:Merged file forcing_obc_segment_004.nc already exists. Skipping.
INFO:CrocoDash.extract_forcings.obc:OBC processing complete.
INFO:CrocoDash.extract_forcings.mom6:Initial condition files already exist. They will be skipped.
INFO:CrocoDash.extract_forcings.mom6:Initial condition filled files already exist. They will be skipped.
INFO:CrocoDash.extract_fo

This means that some areas may only have one or two layers between the surface and sea floor. 
For increased stability, consider increasing the minimum depth, or adjusting the vertical coordinate to add more layers near the surface.


2026-08-07 21:40:40,718 INFO Request ID is 654e5793-3be4-43c8-a53a-29a4e47be7c0
INFO:ecmwf.datastores.legacy_client:Request ID is 654e5793-3be4-43c8-a53a-29a4e47be7c0
2026-08-07 21:40:40,870 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted


KeyboardInterrupt: 

## Inspect the real multi-station ERA5 output

Every real ERA5 grid point in each boundary's buffered window is its own station -- hundreds across the domain's perimeter -- each carrying its own decoded `E(f, theta)` spectrum from a real CDS pull, not a synthetic placeholder.

In [ ]:
wave_dir = INPUTDIR / "wave"

spec_names = (wave_dir / "spec.list").read_text().splitlines()
print("total real ERA5 stations across all 4 boundaries:", len(spec_names))
print(
    [l for l in (wave_dir / "ww3_bounc.nml").read_text().splitlines() if "INTERP" in l][-1]
)

In [ ]:
import xarray as xr

lats, lons, maxes, nonzero = [], [], [], 0
for name in spec_names:
    ds = xr.open_dataset(wave_dir / name, decode_times=False)
    lats.append(float(ds["latitude"].values[0]))
    lons.append(float(ds["longitude"].values[0]))
    maxes.append(float(ds["efth"].max()))
    if (ds["efth"].values != 0).any():
        nonzero += 1
    ds.close()

print(f"stations with nonzero wave energy: {nonzero}/{len(spec_names)}")
print(f"latitude range:  {min(lats):.2f} to {max(lats):.2f}")
print(f"longitude range: {min(lons):.2f} to {max(lons):.2f}")
print(f"peak efth across all stations: {max(maxes):.3f} m^2 s / rad")

## Case is set up

`./case.build`/`./case.submit` are intentionally not run here -- same as the synthetic-data demo, this notebook validates the OBC-generation pipeline. A full 8-day MOM6+WW3 integration against this exact real-ERA5 case has already been run and verified end to end on Derecho (clean restarts, growing significant wave height propagating in from the boundary, real ERA5-shaped spectral peak) -- `./case.build`/`./case.submit` from `CASEROOT` is the natural next step to reproduce that.

In [ ]:
print("caseroot:", case.caseroot)
print("inputdir:", case.inputdir)